# Ingeniería de características - House Pricing

**Autor:** Roberto Farfan

Construye el pipeline de transformaciones (scikit-learn + feature-engine) para las features seleccionadas en `1.0-rf-analisis-exploratorio` (`MODEL_FEATURES` en `pipeline_config.yaml`) y genera los datasets procesados que usará el notebook de modelos.

Pasos:
1. Imputación de variables numéricas y categóricas
2. Codificación de variables categóricas
3. Tratamiento de outliers
4. Transformación de variables continuas
5. Escalado de variables

Cada paso se muestra por separado sobre el set de entrenamiento y al final se unen en un único `Pipeline` de scikit-learn.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from feature_engine.encoding import OneHotEncoder, RareLabelEncoder
from feature_engine.imputation import CategoricalImputer, MeanMedianImputer, RandomSampleImputer
from feature_engine.outliers import Winsorizer
from feature_engine.selection import DropFeatures
from feature_engine.transformation import YeoJohnsonTransformer
from feature_engine.wrappers import SklearnTransformerWrapper
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
import yaml

from house_pricing.config import MODELS_DIR, PROCESSED_DATA_DIR, PROJ_ROOT, RAW_DATA_DIR

2026-11-10 20:14:27.122 | INFO     | house_pricing.config:<module>:12 - PROJ_ROOT path is: C:\Users\rfarfan\OneDrive - Universidad Galileo\2026T4\Product Development\Ejercicios Clase\House_Pricing


## Carga de datos y features

Carga `train.csv` y `test.csv` (con `Id` como índice para poder unir las predicciones después), lee `MODEL_FEATURES` desde `pipeline_config.yaml` y clasifica las features en continuas, discretas y categóricas con el mismo criterio del análisis exploratorio (discreta = 15 valores únicos o menos; `MSSubClass` se mantiene como numérica discreta).

In [3]:
target = "SalePrice"

train = pd.read_csv(RAW_DATA_DIR / "train.csv", index_col="Id")
test = pd.read_csv(RAW_DATA_DIR / "test.csv", index_col="Id")

with open(PROJ_ROOT / "pipeline_config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
model_features = config["MODEL_FEATURES"]

max_valores_discretos = 15
num_features = [col for col in model_features if pd.api.types.is_numeric_dtype(train[col])]
discretas = [col for col in num_features if train[col].nunique() <= max_valores_discretos]
continuas = [col for col in num_features if col not in discretas]
categoricas = [col for col in model_features if col not in num_features]

print(f"train: {train.shape} | test: {test.shape}")
print(f"Continuas ({len(continuas)}): {continuas}")
print(f"Discretas ({len(discretas)}): {discretas}")
print(f"Categóricas ({len(categoricas)}): {categoricas}")

train: (1460, 80) | test: (1459, 79)
Continuas (10): ['GrLivArea', 'GarageArea', 'TotalBsmtSF', '1stFlrSF', 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'MasVnrArea', 'BsmtFinSF1', 'LotFrontage']
Discretas (10): ['OverallQual', 'GarageCars', 'FullBath', 'TotRmsAbvGrd', 'Fireplaces', 'HalfBath', 'BsmtFullBath', 'BedroomAbvGr', 'KitchenAbvGr', 'MSSubClass']
Categóricas (10): ['Neighborhood', 'ExterQual', 'BsmtQual', 'KitchenQual', 'GarageFinish', 'FireplaceQu', 'Foundation', 'GarageType', 'BsmtFinType1', 'HeatingQC']


## Eliminación de outliers extremos

Antes de dividir los datos se eliminan las casas con `GrLivArea > 4000` y `SalePrice < 300000`: son casas muy grandes con un precio anormalmente bajo, y el autor del dataset (De Cock) recomienda quitarlas. Se hace fuera del pipeline porque un `Pipeline` de scikit-learn no puede eliminar filas sin desalinear `X` y `y`.

In [4]:
mask_outliers = (train["GrLivArea"] > 4000) & (train[target] < 300000)
print(f"Casas eliminadas: {mask_outliers.sum()}")
display(train.loc[mask_outliers, ["GrLivArea", target]])

train = train.loc[~mask_outliers]
print(f"train sin outliers extremos: {train.shape}")

Casas eliminadas: 2


,GrLivArea,SalePrice
Id,,
524,4676,184750
1299,5642,160000


train sin outliers extremos: (1458, 80)


## Target y división en entrenamiento y validación

Transforma el target con `log1p` para corregir su sesgo (las predicciones se regresan a dólares con `np.expm1`) y divide los datos en entrenamiento (80 %) y validación (20 %). Todas las transformaciones se ajustan **solo con el set de entrenamiento** para no filtrar información de validación.

In [5]:
X = train[model_features]
y = np.log1p(train[target]).rename("SalePrice_log")
X_test = test[model_features]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Sesgo de SalePrice: {train[target].skew():.2f} -> log1p: {y.skew():.2f}")
print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")

Sesgo de SalePrice: 1.88 -> log1p: 0.12
X_train: (1166, 30) | X_val: (292, 30) | X_test: (1459, 30)


## Paso 1: Imputación

Reglas según el porcentaje de faltantes en el set de entrenamiento:

| % faltantes | Numéricas | Categóricas |
|---|---|---|
| ≤ 5 % | Mediana (`MeanMedianImputer`) | Moda (`CategoricalImputer`) |
| > 5 % y ≤ 20 % | Muestreo aleatorio de los valores observados (`RandomSampleImputer`), que conserva la distribución real | Moda |
| > 20 % | Se descarta la variable (`DropFeatures`) | Se descarta la variable |

Las variables sin faltantes en entrenamiento también reciben el imputador de mediana o moda, porque `test.csv` sí tiene faltantes en algunas de ellas.

In [6]:
pct_faltantes = X_train.isna().mean() * 100

descartar = pct_faltantes[pct_faltantes > 20].index.tolist()
num_aleatorio = [col for col in num_features if 5 < pct_faltantes[col] <= 20]
num_mediana = [col for col in num_features if pct_faltantes[col] <= 5]
cat_moda = [col for col in categoricas if col not in descartar]

display(pct_faltantes[pct_faltantes > 0].sort_values(ascending=False).round(2).to_frame("pct_faltantes"))
print(f"Descartar (> 20 %): {descartar}")
print(f"Muestreo aleatorio (5-20 %): {num_aleatorio}")
print(f"Mediana (<= 5 %): {len(num_mediana)} variables numéricas")
print(f"Moda: {len(cat_moda)} variables categóricas")

# Actualiza los grupos sin las variables descartadas
continuas = [col for col in continuas if col not in descartar]
discretas = [col for col in discretas if col not in descartar]
categoricas = cat_moda

pasos_imputacion = [
    ("descartar", DropFeatures(features_to_drop=descartar)),
    ("imputar_aleatorio", RandomSampleImputer(variables=num_aleatorio, random_state=42)),
    ("imputar_mediana", MeanMedianImputer(imputation_method="median", variables=num_mediana)),
    ("imputar_moda", CategoricalImputer(imputation_method="frequent", variables=cat_moda)),
]

X_paso = Pipeline(pasos_imputacion).fit_transform(X_train)
print(f"Faltantes después de imputar: {X_paso.isna().sum().sum()} | columnas: {X_paso.shape[1]}")

,pct_faltantes
FireplaceQu,47.34
LotFrontage,18.27
GarageFinish,5.83
GarageYrBlt,5.83
GarageType,5.83
BsmtQual,2.74
BsmtFinType1,2.74
MasVnrArea,0.51


Descartar (> 20 %): ['FireplaceQu']
Muestreo aleatorio (5-20 %): ['GarageYrBlt', 'LotFrontage']
Mediana (<= 5 %): 18 variables numéricas
Moda: 9 variables categóricas
Faltantes después de imputar: 0 | columnas: 29


## Paso 2: Codificación de variables categóricas

- **Ordinales:** variables con un orden natural. Se codifican respetando ese orden (0 = peor):
  - Calidad (`ExterQual`, `BsmtQual`, `KitchenQual`, `HeatingQC`): `Po < Fa < TA < Gd < Ex`
  - `GarageFinish`: `Unf < RFn < Fin`
  - `BsmtFinType1`: `Unf < LwQ < Rec < BLQ < ALQ < GLQ`
- **Nominales:** sin orden (`Neighborhood`, `Foundation`, `GarageType`). Primero las categorías con menos del 5 % de las casas se agrupan en `Rare` (`RareLabelEncoder`) y después se aplica one-hot (`OneHotEncoder`).

In [7]:
escala_calidad = ["Po", "Fa", "TA", "Gd", "Ex"]
orden_ordinales = {
    "ExterQual": escala_calidad,
    "BsmtQual": escala_calidad,
    "KitchenQual": escala_calidad,
    "HeatingQC": escala_calidad,
    "GarageFinish": ["Unf", "RFn", "Fin"],
    "BsmtFinType1": ["Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
}
ordinales = [col for col in categoricas if col in orden_ordinales]
nominales = [col for col in categoricas if col not in orden_ordinales]
print(f"Ordinales: {ordinales}")
print(f"Nominales: {nominales}")

pasos_codificacion = [
    ("codificar_ordinales", SklearnTransformerWrapper(
        transformer=OrdinalEncoder(
            categories=[orden_ordinales[col] for col in ordinales],
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        ),
        variables=ordinales,
    )),
    ("agrupar_raras", RareLabelEncoder(tol=0.05, n_categories=1, variables=nominales)),
    ("one_hot", OneHotEncoder(variables=nominales)),
]

X_paso = Pipeline(pasos_codificacion).fit_transform(X_paso)
print(f"Columnas después de codificar: {X_paso.shape[1]}")
X_paso.head()

Ordinales: ['ExterQual', 'BsmtQual', 'KitchenQual', 'GarageFinish', 'BsmtFinType1', 'HeatingQC']
Nominales: ['Neighborhood', 'Foundation', 'GarageType']
Columnas después de codificar: 44


,GrLivArea,GarageArea,TotalBsmtSF,1stFlrSF,YearBuilt,YearRemodAdd,GarageYrBlt,MasVnrArea,BsmtFinSF1,LotFrontage,...,Neighborhood_NridgHt,Neighborhood_OldTown,Foundation_CBlock,Foundation_PConc,Foundation_BrkTil,Foundation_Rare,GarageType_Attchd,GarageType_Detchd,GarageType_BuiltIn,GarageType_Rare
Id,,,,,,,,,,,,,,,,,,,,,
255,1314,294,1314,1314,1957,1957,1957.0,0.0,922,70.0,...,0,0,1,0,0,0,1,0,0,0
1067,1571,380,799,799,1993,1994,1993.0,0.0,0,59.0,...,0,0,0,1,0,0,1,0,0,0
866,1002,902,1002,1002,1970,1970,1973.0,76.0,828,70.0,...,0,0,1,0,0,0,0,1,0,0
800,1768,240,731,981,1937,1950,1939.0,252.0,569,60.0,...,0,0,0,0,1,0,0,1,0,0
381,1691,308,1026,1026,1924,1950,1924.0,0.0,218,50.0,...,0,0,0,0,1,0,0,1,0,0


## Paso 3: Tratamiento de outliers

Recorta (capping) las variables continuas a los límites del rango intercuartílico: `Q1 - 1.5·IQR` y `Q3 + 1.5·IQR` (`Winsorizer`). Los valores fuera de esos límites se reemplazan por el límite, sin eliminar filas.

In [8]:
pasos_outliers = [
    ("capping", Winsorizer(capping_method="iqr", tail="both", fold=1.5, variables=continuas)),
]

pipe_outliers = Pipeline(pasos_outliers)
X_antes = X_paso
X_paso = pipe_outliers.fit_transform(X_paso)

winsorizer = pipe_outliers.named_steps["capping"]
limites = pd.DataFrame({
    "limite_inferior": winsorizer.left_tail_caps_,
    "limite_superior": winsorizer.right_tail_caps_,
    "pct_recortados": ((X_antes[continuas] != X_paso[continuas]).mean() * 100).round(2),
})
limites

,limite_inferior,limite_superior,pct_recortados
GrLivArea,154.000,2768.000,1.89
GarageArea,-45.875,949.125,1.46
TotalBsmtSF,76.000,1996.000,4.80
1stFlrSF,153.375,2112.375,1.54
YearBuilt,1885.000,2069.000,0.51
YearRemodAdd,1911.500,2059.500,0.00
GarageYrBlt,1902.000,2062.000,0.09
MasVnrArea,-242.625,404.375,6.43
BsmtFinSF1,-1073.250,1788.750,0.34
LotFrontage,27.500,111.500,6.95


## Paso 4: Transformación de variables continuas

Aplica Yeo-Johnson (`YeoJohnsonTransformer`) a las variables continuas con sesgo alto (`|skew| > 0.75`, medido después del capping). Yeo-Johnson acepta ceros y valores negativos, a diferencia de Box-Cox o del logaritmo.

In [9]:
umbral_sesgo = 0.75
sesgo = X_paso[continuas].skew()
sesgadas = sesgo[sesgo.abs() > umbral_sesgo].index.tolist()
print(f"Variables a transformar: {sesgadas}")

pasos_transformacion = [
    ("yeo_johnson", YeoJohnsonTransformer(variables=sesgadas)),
]

X_paso = Pipeline(pasos_transformacion).fit_transform(X_paso)

pd.DataFrame({
    "sesgo_antes": sesgo[sesgadas],
    "sesgo_despues": X_paso[sesgadas].skew(),
}).round(2)

Variables a transformar: ['MasVnrArea']


,sesgo_antes,sesgo_despues
MasVnrArea,1.29,0.43


## Paso 5: Escalado

Estandariza todas las columnas finales con `StandardScaler` (media 0 y desviación estándar 1), para que ninguna variable domine por su escala en modelos lineales o basados en distancias.

In [10]:
pasos_escalado = [
    ("escalado", SklearnTransformerWrapper(transformer=StandardScaler())),
]

X_paso = Pipeline(pasos_escalado).fit_transform(X_paso)
X_paso.describe().T[["mean", "std"]].round(2).head(10)

,mean,std
GrLivArea,0.0,1.0
GarageArea,0.0,1.0
TotalBsmtSF,0.0,1.0
1stFlrSF,-0.0,1.0
YearBuilt,0.0,1.0
YearRemodAdd,-0.0,1.0
GarageYrBlt,0.0,1.0
MasVnrArea,0.0,1.0
BsmtFinSF1,0.0,1.0
LotFrontage,-0.0,1.0


## Pipeline final

Une los cinco pasos en un único `Pipeline` de scikit-learn, lo ajusta con el set de entrenamiento y transforma entrenamiento, validación y el `test.csv` de Kaggle.

In [11]:
pipeline_fe = Pipeline(
    pasos_imputacion
    + pasos_codificacion
    + pasos_outliers
    + pasos_transformacion
    + pasos_escalado
)

X_train_fe = pipeline_fe.fit_transform(X_train)
X_val_fe = pipeline_fe.transform(X_val)
X_test_fe = pipeline_fe.transform(X_test)

for nombre, datos in [("X_train", X_train_fe), ("X_val", X_val_fe), ("X_test", X_test_fe)]:
    print(f"{nombre}: {datos.shape} | faltantes: {datos.isna().sum().sum()}")

pipeline_fe

X_train: (1166, 44) | faltantes: 0
X_val: (292, 44) | faltantes: 0
X_test: (1459, 44) | faltantes: 0


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('descartar', ...), ('imputar_aleatorio', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0",list,"['Gr...ea', 'Ga...ea', 'To...SF', '1s...SF', ...]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,features_to_drop,['FireplaceQu']
Name,Type,Value
feature_names_in_,list,"['Gr...ea', 'Ga...ea', 'To...SF', '1s...SF', ...]"
features_to_drop_,list,['Fi...Qu']


## Guardar datasets y pipeline

Guarda los datasets procesados en `data/processed/` (con `Id` como índice) y el pipeline ajustado en `models/`, para usarlos en el notebook de modelos. El target se guarda en escala logarítmica (`SalePrice_log`).

In [12]:
X_train_fe.to_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_val_fe.to_csv(PROCESSED_DATA_DIR / "X_val.csv")
X_test_fe.to_csv(PROCESSED_DATA_DIR / "X_test.csv")
y_train.to_csv(PROCESSED_DATA_DIR / "y_train.csv")
y_val.to_csv(PROCESSED_DATA_DIR / "y_val.csv")

joblib.dump(pipeline_fe, MODELS_DIR / "pipeline_fe.joblib")

print(sorted(p.name for p in PROCESSED_DATA_DIR.glob("*.csv")))
print(sorted(p.name for p in MODELS_DIR.glob("*.joblib")))

['X_test.csv', 'X_train.csv', 'X_val.csv', 'predicciones_test.csv', 'y_train.csv', 'y_val.csv']
['modelo_ganador.joblib', 'pipeline_fe.joblib', 'pipeline_inferencia.joblib']
